# grad-expressed-in-out — ex3: softplus_back via cached out — third activation, no recompute from x

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `grad-expressed-in-out`. Running the final beacon cell reports progress against the `Backprop: grad expressed in out` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: grad expressed in out` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grad-expressed-in-out`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grad-expressed-in-out"
DD_SUBTOPIC = "Backprop: grad expressed in out"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## softplus_back via cached out — third activation, same pattern

Ex1 wrote `sigmoid_back = grad_out * out * (1 - out)`. Ex2 wrote
`tanh_back = grad_out * (1 - out**2)`. The deepening move applies the
SAME 'grad expressed in `out`' template to a THIRD activation —
`softplus(x) = log(1 + exp(x))` — to confirm you've internalized the
pattern rather than memorized two specific formulas.

**Math.**
```
out = softplus(x) = log(1 + exp(x))

d/dx softplus(x) = exp(x) / (1 + exp(x))
                 = sigmoid(x)
                 = 1 - exp(-out)        ← expressed in out
```

The last step uses `exp(-out) = exp(-log(1+exp(x))) = 1/(1+exp(x))`,
so `1 - exp(-out) = exp(x)/(1+exp(x)) = sigmoid(x)`.

**Chain rule.**
```
dL/dx = grad_out * (1 - t.exp(-out))
```

**Why this transfer test matters.** Sigmoid and tanh have algebraically
obvious closed forms in `out`. Softplus needs ONE rewrite to express
the derivative in `out` — and that rewrite (`sigmoid(x) = 1 -
exp(-softplus(x))`) is the test of whether you understood why caching
`out` is even useful. The drill explicitly forbids calling `t.sigmoid`
or recomputing `softplus` from `x`.

### Exercise 3 — softplus_back via cached out — third activation, no recompute from x

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the 'grad expressed in `out`' pattern to softplus by writing `softplus_back(grad_out, out, x) = grad_out * (1 - t.exp(-out))`, reusing the cached forward output rather than recomputing `softplus(x)` or invoking `t.sigmoid(x)`.
> Keywords: softplus, cached-out, elementwise, no-recompute, transfer
> ```

**KCs targeted:** `grad-expressed-in-out`, `back-fn-uses-cached-out`

Implement `ex3_softplus_back(grad_out, out, x)` using ONLY the cached `out` — no `t.sigmoid(x)`, no `t.softplus(x)`, no `t.exp(x)` on the raw input.

**Math.** `out = softplus(x) = log(1 + exp(x))`. The derivative is `sigmoid(x)`, which equals `1 - exp(-out)`:

```
exp(-out) = exp(-log(1+exp(x))) = 1 / (1+exp(x))
1 - exp(-out) = exp(x) / (1+exp(x)) = sigmoid(x)
```

So by the chain rule:

```
dL/dx = grad_out * (1 - t.exp(-out))
```

**The drill's constraint.** The function MUST be expressible in ONE line that uses `out` and not `x`. The test will pass a deliberately WRONG `x` (zeros, or unrelated values) to catch any attempt to recompute from `x`.

Inputs:
- `grad_out`: `Tensor`, dL/d(out), same shape as `out`.
- `out`: `Tensor`, the cached `softplus(x_real)` from forward.
- `x`: `Tensor`, the original input — passed for signature compatibility but UNUSED.

Output: `Tensor` of the same shape, `dL/dx`.

In [ ]:
def ex3_softplus_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    """dL/dx for softplus, via cached out: grad_out * (1 - exp(-out))."""
    raise NotImplementedError()


def _test_ex3():
    def _test_ex3():
        import torch.nn.functional as F

        # === Reference path: autograd-derived sigmoid(x) ===
        x = t.tensor([-2.0, -0.5, 0.0, 0.5, 2.0], requires_grad=True)
        out = F.softplus(x)
        ref_grad = t.sigmoid(x).detach()    # d/dx softplus = sigmoid
        grad_out = t.tensor([1.0, 1.0, 1.0, 1.0, 1.0])

        got = ex3_softplus_back(grad_out, out.detach(), x.detach())
        assert t.allclose(got, ref_grad, atol=1e-6), f'softplus_back mismatch: got {got}, ref {ref_grad}'

        # === Chain rule: grad_out * sigmoid(x) ===
        grad_out = t.tensor([0.5, 2.0, 1.0, -1.0, 3.0])
        got = ex3_softplus_back(grad_out, out.detach(), x.detach())
        expected = grad_out * ref_grad
        assert t.allclose(got, expected, atol=1e-6), got

        # === MUST use out, not x: pass WRONG x, output must still be correct. ===
        grad_out = t.ones(5)
        bogus_x = t.zeros(5)            # NOT the real x — would give sigmoid(0)=0.5 everywhere
        got = ex3_softplus_back(grad_out, out.detach(), bogus_x)
        assert t.allclose(got, ref_grad, atol=1e-6), (
            f'function must use out, not x; bogus x produced wrong result: {got} vs ref {ref_grad}'
        )

        # === Scalar input ===
        x_s = t.tensor(1.5, requires_grad=True)
        out_s = F.softplus(x_s).detach()
        ref_s = t.sigmoid(t.tensor(1.5))
        got_s = ex3_softplus_back(t.tensor(1.0), out_s, x_s.detach())
        assert abs(got_s.item() - ref_s.item()) < 1e-6, f'scalar mismatch: {got_s.item()} vs {ref_s.item()}'

        # === Multi-dim input ===
        x_m = t.randn(3, 4, requires_grad=True)
        t.manual_seed(0)
        x_m = t.randn(3, 4, requires_grad=True)
        out_m = F.softplus(x_m)
        ref_m = t.sigmoid(x_m).detach()
        g_m = t.randn(3, 4)
        got_m = ex3_softplus_back(g_m, out_m.detach(), x_m.detach())
        assert t.allclose(got_m, g_m * ref_m, atol=1e-6), got_m

        # === Large positive x: softplus(x) ≈ x, exp(-out) ≈ 0, derivative ≈ 1 ===
        x_big = t.tensor([10.0, 20.0, 30.0], requires_grad=True)
        out_big = F.softplus(x_big).detach()
        got_big = ex3_softplus_back(t.ones(3), out_big, x_big.detach())
        assert t.allclose(got_big, t.ones(3), atol=1e-3), f'large-x derivative should be ~1: {got_big}'

        # === Large negative x: softplus(x) ≈ 0, exp(-out) ≈ 1, derivative ≈ 0 ===
        x_neg = t.tensor([-10.0, -20.0, -30.0], requires_grad=True)
        out_neg = F.softplus(x_neg).detach()
        got_neg = ex3_softplus_back(t.ones(3), out_neg, x_neg.detach())
        assert t.allclose(got_neg, t.zeros(3), atol=1e-3), f'large-negative-x derivative should be ~0: {got_neg}'
        print('ex3 ok')

    _test_ex3()
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_softplus_back(grad_out, out, x):
    # Chain rule: dL/dx = grad_out * d/dx softplus(x)
    #          = grad_out * sigmoid(x)
    #          = grad_out * (1 - exp(-out))     (since out = log(1+exp(x)))
    return grad_out * (1 - t.exp(-out))
```

**Why express sigmoid(x) as `1 - exp(-out)`.** The whole point of caching `out` from the forward pass is to avoid recomputing the expensive transcendental on the backward pass. `t.sigmoid(x)` would work numerically but defeats the purpose — it runs another exp+division for every element.

**Numerical stability of `1 - exp(-out)`.** For large positive `x`, `out ≈ x` (large), so `exp(-out) ≈ 0` and the derivative saturates to 1 — exactly correct. For large negative `x`, `out ≈ 0`, so `exp(-out) ≈ 1` and the derivative is ≈ 0 — also correct. The formula stays well-conditioned across the input range.

**This is the third activation in the family.** Sigmoid: `out * (1 - out)`. Tanh: `1 - out**2`. Softplus: `1 - exp(-out)`. Each closed form is one line in `out` — the test is whether you can DERIVE the third without being given the formula. The recap spells it out; the ability to internalize that derivation is the actual learning objective.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()